<h1 style="font-size: 1.6rem; font-weight: bold">ITO5163: Introduction to Cybersecurity</h1>
<h1 style="font-size: 1.6rem; font-weight: bold">Module 5: Attack Models</h1>
<p style="margin-top: 5px; margin-bottom: 5px;">Monash University Australia</p>
<p style="margin-top: 5px; margin-bottom: 5px;">Jupyter Notebook by: Tristan Sim Yook Min</p>
References: Information Source from Monash Faculty of Information Technology

---

### **Security Models**

Every security mechanism is built on an **algorithm** or a **hard mathematical problem**. The maths may be sound, but how it's **used** can still have weaknesses, now or in the future. So security mechanisms have to be **evaluated** for how much security they really offer, mostly through **cryptanalysis**: the branch of cryptology that studies how cryptographic algorithms can be broken.

- **Two Things to Consider**:
  - **The algorithm itself**: how it works, and what attacks might break it
  - **Time**: computers keep getting faster and attacks keep improving, so a problem that's too hard to solve today may become solvable later

There are five common models for evaluating security, from strongest to weakest guarantee:

```mermaid
flowchart LR
    U["Unconditional security<br>(unlimited attacker)"] --> P["Provable security<br>(as hard as a known problem)"] --> C["Computational security<br>(best known attacks too costly)"] --> A["Ad hoc security<br>(convincing arguments only)"]
```

**Complexity-theoretic security** is a theoretical framework rather than a point on this scale.

| Model | Attacker Assumed to Have | Security Means | Example |
|-------|--------------------------|----------------|---------|
| **Unconditional** | **Unlimited** computing power | The ciphertext reveals **nothing**, even with infinite computing | One-time pad |
| **Complexity-theoretic** | **Polynomial-time** computing | Mathematical proof using asymptotic, worst-case analysis | Theoretical research |
| **Provable** | Realistic computing | Breaking it is **as hard as** a well-known hard problem | RSA-OAEP, Goldwasser-Micali |
| **Computational** (practical) | The **best known attacks** | The effort needed is **far beyond** any attacker's resources | AES, SHA-256 |
| **Ad hoc** (heuristic) | Whatever the designers considered | **Convincing arguments**, but no proof | Many new or proprietary designs |

#### **Unconditional Security**

The **strongest** model. The attacker is assumed to have **unlimited computing power**, and the question is whether there's **enough information** to break the system at all. For encryption, this is called **perfect secrecy**: observing **any amount** of ciphertext reveals **nothing** about the plaintext.

- **Requirements for Perfect Secrecy**:
  - The key must be **at least as long as the message**
  - The key must be **truly random** and **never reused**
  - Usually **impractical**: sharing a key as long as every message you'll ever send is hard

- **Systems With Shorter Keys**: they **can't** be unconditionally secure. In theory, every bit of ciphertext observed reduces the uncertainty about the plaintext and key
- **Public-key systems** can't be unconditionally secure either: an attacker with unlimited power could encrypt every possible plaintext with the public key and see which one matches
- **Example: the One-Time Pad**

| Ciphertext | Key | Plaintext |
|------------|-----|-----------|
| `53114b724b01` | `12451f33084a` | `ATTACK` |
| `53114b724b01` | `17540d370545` | `DEFEND` |
| `53114b724b01` | `01541f3b1944` | `RETIRE` |

Every 6-letter message is possible with **some** key, so even trying every key tells the attacker nothing.

#### **Complexity-Theoretic Security**

A **theoretical** model. The attacker can use any attack whose time and memory grow **polynomially** with the security parameter (such as the key size). A **security proof** is built from this.

- **Features**:
  - Aims to build cryptography on the **weakest assumptions possible**
  - Uses **asymptotic** and **worst-case** analysis
  - Important for **theory**, but doesn't give concrete answers like "how many bits of key are needed"
  - A "polynomial-time" attack could still be **infeasible in practice**, e.g. one taking $n^{50}$ steps

#### **Provable Security**

A system is **provably secure** if breaking it is shown to be **at least as hard as** solving a well-known **hard problem**, such as:

- **Factoring** the product of two large primes (as in RSA)
- Solving the **discrete logarithm** problem (as in Diffie-Hellman)

- **Features**:
  - The proof is a **reduction**: "if you could break this system, you could also solve the hard problem"
  - "Provable" is **conditional**: the system is only as secure as the hard problem really is
  - Widely seen as the **preferred** approach for practical analysis

#### **Computational Security (Practical Security)**

The system is studied extensively to find the **best known attacks**. If even those attacks need **far more computing effort** than any attacker could have, the system is **computationally secure**.

- **Features**:
  - The effort must exceed attackers' resources **by a large margin**
  - The attacks considered must be the **best available**
  - **Most cryptosystems in use today**, both symmetric and public-key, fall into this category
  - Security can **weaken over time** as computers and attacks improve

- **Example: brute-forcing AES-128** at a billion billion ($10^{18}$) keys per second:

$$\frac{2^{128}}{10^{18}} \approx 3.4 \times 10^{20} \text{ seconds} \approx 1.1 \times 10^{13} \text{ years}$$

That's about **780 times the age of the universe**.

#### **Ad Hoc Security (Heuristic Security)**

The system is argued to be secure with **convincing arguments**, e.g. that an attacker would need far more time or memory than they could have. Systems that pass are said to have **heuristic security**.

- **Features**:
  - **No formal proof**, only arguments
  - **Attacks that weren't considered** can still break the system
  - The weakest model, so claims of security should be treated with caution

<br>

***

### **Attacks on Encryption Schemes**

Encryption schemes depend on keeping the **key secret**. Many **cryptanalytic attacks** target this weak point, trying to **recover the plaintext** from captured ciphertext or, in the worst case, **work out the key itself**.

Encrypted data stored on a computer that's disconnected from any network is fairly safe, but that's not realistic. Modern systems constantly **send encrypted data over networks**, which gives attackers many more opportunities.

#### **Network Security Model**

```mermaid
flowchart LR
    M1["Sender's<br>message"] --> T1(("Security<br>transformation")) --> CH[["Information channel<br>(secure message)"]] --> T2(("Security<br>transformation")) --> M2["Recipient's<br>message"]
    TTP["Trusted third party<br>(e.g. distributes secret information)"] -. "secret information" .-> T1
    TTP -. "secret information" .-> T2
    OPP["Opponent"] <--> CH
```

- **The Model**:
  - The **sender** transforms the message (e.g. encrypts it) using **secret information** (e.g. a key)
  - The **secure message** travels over the **information channel**
  - The **recipient** reverses the transformation using their secret information
  - A **trusted third party** may distribute the secret information, or settle disputes
  - The **opponent** tries to attack, usually through the channel

- **Why the Channel Is the Weak Point**:
  - Attackers target whatever they can access **freely** while staying **hidden** the longest
  - The channel is easy to reach: **sniffing** network traffic takes little effort and is hard to detect
  - Once the attacker has collected enough ciphertext, they can work on **deciphering** it

- **Four Basic Tasks of a Security Service** (each a possible target for an attacker):
  - Design an **algorithm** that turns plaintext into secure ciphertext
  - Generate the **secret information** (keys) the algorithm needs
  - Develop methods to **distribute and share** the secret information
  - Define a **protocol** so both parties can use the algorithm and secret information to communicate

#### **Types of Cryptanalytic Attack**

The attacks differ in **how much the attacker knows or can do**. In all of them, the attacker is assumed to **know the algorithm**, and the only secret is the key.

| Attack | Attacker Has | Attacker Can Do |
|--------|--------------|-----------------|
| **Ciphertext-only (COA)** | Ciphertext | Analyse it |
| **Known-plaintext (KPA)** | Ciphertext, plus some plaintext-ciphertext pairs | Compare them |
| **Chosen-plaintext (CPA)** | Access to the **encryption** process | Encrypt plaintexts of their choice, chosen **in advance** |
| **Adaptive chosen-plaintext (ACPA)** | Access to the encryption process | Choose each new plaintext **based on earlier results** |
| **Chosen-ciphertext (CCA)** | Access to the **decryption** process | Decrypt ciphertexts of their choice, chosen in advance |
| **Adaptive chosen-ciphertext (ACCA)** | Access to the decryption process | Choose each new ciphertext **based on earlier results** |

Each attack in the table gives the attacker **more power** than the one above it. A good modern cipher should resist **all** of them.

#### **Ciphertext-Only Attack (COA)**

The attacker only has the **ciphertext** (and knows the algorithm). This is the **hardest** position for an attacker, since they know nothing about the plaintext.

- **How It Works**:
  - Capture **as much ciphertext as possible**
  - Run **statistical analysis**, e.g. letter and **digram** frequencies in the language
  - Very effective against **classical ciphers** (Caesar, Vigenère, Playfair), which leave language patterns in the ciphertext
  - Modern ciphers like **AES**, used in a proper mode, are designed so the ciphertext shows **no usable patterns**

#### **Known-Plaintext Attack (KPA)**

The attacker knows the algorithm, has the ciphertext, and also has **one or more plaintext-ciphertext pairs** encrypted with the **same key**.

- **How It Works**:
  - Attackers often know **part of a message** in advance, because systems use **predictable formats**
  - Example: a system always starts its message with "Login", then asks for the password. Knowing that part helps the attacker work out the **rest**, such as the password
  - Historical example: during WWII, codebreakers used predictable phrases in **Enigma** messages, like weather reports, called "**cribs**"
  - If a COA reveals enough **patterns** to guess some plaintext, it can turn into a KPA

#### **Chosen-Plaintext Attack (CPA)**

The attacker gets access to the **encryption** process, e.g. a device **left unattended**, or an **insider** at the organisation. They encrypt **plaintexts of their choice** and study the results to work out the key.

- **Notes**:
  - In a basic CPA, the attacker chooses all the plaintexts **at the start**
  - For **public-key systems**, anyone can encrypt with the public key, so a CPA is **always possible**

#### **Adaptive Chosen-Plaintext Attack (ACPA)**

Like CPA, but the attacker chooses plaintexts **one at a time**, using the results so far to pick the **next** one. Each result guides the next choice, so it's **more powerful** than a basic CPA.

#### **Chosen-Ciphertext Attack (CCA)**

The attacker gets access to the **decryption** process. They choose **ciphertexts** to decrypt and study the plaintexts produced, to work out the key or decrypt other messages.

- **Notes**:
  - Applies to **both** symmetric and public-key systems, but is especially important for **public-key** systems
  - Sometimes called a "**lunchtime attack**": the attacker uses an unattended decryption device while its owner is away

#### **Adaptive Chosen-Ciphertext Attack (ACCA)**

Like CCA, but the attacker chooses ciphertexts **one at a time**, using each decrypted result to pick the **next** ciphertext.

- **Notes**:
  - The **strongest** attack in this list
  - Real example: **Bleichenbacher's attack (1998)** sent thousands of modified ciphertexts to SSL servers. From whether each one was accepted or rejected, it gradually recovered **RSA-encrypted** secrets
  - Modern schemes such as **RSA-OAEP** and **AES-GCM** are designed to resist ACCA

#### **Beyond Encryption**

These attack types also apply to other cryptographic tools, such as **digital signatures** (where the attacker can get messages of their choice signed), **MAC algorithms**, and **pseudorandom number generators**.

<br>

***

### **Attacks on Protocols**

Attackers don't only target the encryption algorithm. If the **network** or the **protocol** is the weakest link, they'll attack that instead. A strong cipher can still be defeated by a badly designed protocol.

| Attack | Targets | Main Defence |
|--------|---------|--------------|
| **Known-key** | Session keys | Make every session key independent of the others |
| **Replay** | Authentication messages | Nonces, timestamps, sequence numbers |
| **Impersonation** | Authentication | Strong authentication, e.g. MFA and certificates |
| **Dictionary** | Passwords | Unique salts, slow hashing, lockouts |
| **Forward search** | Small sets of possible plaintexts | Random padding (probabilistic encryption) |
| **Interleaving** | Protocol message structure | Make messages unique to their step and session |

#### **Known-Key Attack**

The attacker learns some **old keys** used by the system, and uses them to work out **new keys**.

- **Known-Key Attack**:
  - Works when session keys are **related**, e.g. each new key is derived from the last
  - A good protocol makes every session key **independent**, so learning one old key reveals nothing about the others

- **Related-Key Attack** (a variation):
  - The attacker observes the cipher working under **several different keys** whose **relationship** is known, e.g. keys that differ in just one bit
  - Example: Wi-Fi's old **WEP** standard built each packet's RC4 key by joining a short changing value to the same secret key. The related keys let attackers recover the secret key

#### **Replay Attack**

A **replay** (or **playback**) attack captures packets containing **authentication information**, then **resends** them, unchanged or slightly changed, to get the same access as the real sender.

```mermaid
sequenceDiagram
    participant A as A
    participant E as Attacker E
    participant B as B
    A->>B: Login packet
    Note over E: Records the packet
    Note over E: Later...
    E->>B: Same login packet, unchanged
    B->>E: Access granted (no freshness check)
```

- **Replay Attacks**:
  - Can happen **long after** the original packets were sent
  - The attacker **doesn't need to decrypt** anything
  - Protocols are vulnerable if they don't check that each message is **new**
  - **Defences**: **sequence numbers**, **timestamps**, **nonces**, and **MACs** that cover them

#### **Impersonation (Masquerading) Attack**

The attacker **pretends to be someone else** to get that person's access rights and privileges. These attacks target weaknesses in **authentication**.

- **Ways to Impersonate**:
  - **Stolen credentials**: using someone's real username and password. Hard to detect, since the login looks normal
  - **Spoofing**: faking an **IP address** or **MAC address** to make the network think the attacker's device is the target device
- **Defences**: multi-factor authentication, certificates, and never trusting an IP or MAC address alone as proof of identity

#### **Dictionary Attack**

Mainly targets **passwords**. Usernames are often easy to find, so the password is the real barrier. Systems store passwords as **hashes**.

- **How It Works**:
  - Hashing guesses during an attack takes time, so attackers **precompute** the hashes of likely passwords beforehand
  - Any stolen hash that appears in the precomputed list reveals the password immediately

- **Defences**:
  - **Salting**: a unique, random salt per user makes precomputed lists useless
    - The salt is stored next to the hash, so it isn't secret. What matters is that it's **unique** and **not known before the attack**. A single fixed salt for every user would let the attacker precompute again
  - **Slow hash functions** (bcrypt, Argon2): make each guess expensive, even when the attacker has the salt
  - **Lockouts and timeouts**: stop repeated guessing at the **login page**. These don't help once the hash database itself has been **stolen**, which is why salting and slow hashing are needed too

#### **Forward Search Attack**

Like a dictionary attack, but against **encrypted messages** that can only take a **small number of values**.

- **Example**:
  - A bank encrypts a **32-bit transaction ID** with a **public key**
  - The public key is public, so **anyone** can encrypt with it
  - There are only $2^{32} = 4{,}294{,}967{,}296$ possible IDs
  - The attacker encrypts **every possible ID** with the public key, and compares each result with the captured ciphertext
  - A match reveals the transaction ID, **without ever finding the private key**
  - At 100,000 encryptions per second, trying all $2^{32}$ values takes about **12 hours**

- **Defence**: add **random padding** before encrypting (as in RSA-OAEP), so the same ID encrypts to a **different ciphertext** every time. This is **probabilistic encryption** (lesson 7.4)

#### **Interleaving Attack**

The attacker runs **two or more sessions at the same time** and uses messages from one to **impersonate** someone in the other.

**A flawed protocol.** Here $\gamma$ is a random number (nonce), and $s_X(\ldots)$ means signed by $X$:

```mermaid
sequenceDiagram
    participant A as A
    participant B as B
    A->>B: 1. γA
    B->>A: 2. γB, sB(γB, γA, A)
    A->>B: 3. γ′A, sA(γ′A, γB, B)
```

$$1.\ A \rightarrow B : \gamma_A$$

$$2.\ B \rightarrow A : \gamma_B,\ s_B(\gamma_B, \gamma_A, A)$$

$$3.\ A \rightarrow B : \gamma'_A,\ s_A(\gamma'_A, \gamma_B, B)$$

The idea is that each side's random number, plus the signatures, guarantee **freshness** and **authenticate** both sides. But messages 2 and 3 have the **same structure**: a new random number, then a signature on (new number, other's number, other's name).

**The attack.** E pretends to be A to B, and uses **A itself** to produce the signature it needs:

```mermaid
sequenceDiagram
    participant A as A
    participant E as Attacker E
    participant B as B
    E->>B: 1. γA (E pretends to be A)
    B->>E: 2. γB, sB(γB, γA, A)
    Note over E: E needs A's signature on γB.<br>It starts a second session with A, pretending to be B
    E->>A: 1′. γB (E pretends to be B)
    A->>E: 2′. γ′A, sA(γ′A, γB, B)
    Note over E: A's message 2′ has the same form as message 3
    E->>B: 3. γ′A, sA(γ′A, γB, B)
    Note over B: Accepts E as A
```

| Step | Message | What Happens |
|------|---------|--------------|
| 1 | $E(A) \rightarrow B : \gamma_A$ | E starts a session with B, **pretending to be A** |
| 2 | $B \rightarrow E(A) : \gamma_B,\ s_B(\gamma_B, \gamma_A, A)$ | B replies. To finish, E needs **A's signature** on $\gamma_B$ |
| 1′ | $E(B) \rightarrow A : \gamma_B$ | E starts a **second session** with A, **pretending to be B**, and sends B's nonce |
| 2′ | $A \rightarrow E(B) : \gamma'_A,\ s_A(\gamma'_A, \gamma_B, B)$ | A signs $\gamma_B$, exactly what E needs |
| 3 | $E(A) \rightarrow B : \gamma'_A,\ s_A(\gamma'_A, \gamma_B, B)$ | E forwards A's message as message 3. B **accepts E as A** |

- **Why It Works**: A's message 2′ in one session can be reused as message 3 in another, because they **look the same**
- **Defences**:
  - Make each message's structure **unique to its step**, e.g. include the step number or direction
  - In message 3, sign the **original** nonce: $s_A(\gamma_A, \gamma_B, B)$. Then a signature from a different session won't match

<br>

***

### **Other Attacks**

This lesson covers other kinds of cryptanalysis. Some are **cryptographic**, attacking the maths. Others are **non-cryptographic**, attacking how the system is **built or used**.

| Attack | Type | Targets |
|--------|------|---------|
| Brute force | Cryptographic | Every possible key or password |
| Birthday | Cryptographic | Hash function collisions |
| Side-channel | Non-cryptographic | Physical leaks from the device: time, power, radiation, sound |
| Differential cryptanalysis | Cryptographic | Block ciphers, using chosen plaintexts |
| Linear cryptanalysis | Cryptographic | Block ciphers, using known plaintexts |

#### **Brute Force Attacks**

A **brute force** attack tries **every possible value** until one works. Given enough time, it's **guaranteed to succeed** against any key of fixed length (the one-time pad is the exception). The problem is the **time** it takes.

| Key Size | Example | Time to Try Every Key at $10^{12}$ Keys per Second |
|----------|---------|----------------------------------------------------|
| 56 bits  | DES     | About **20 hours** |
| 64 bits  |         | About **7 months** |
| 80 bits  |         | About **38,000 years** |
| 128 bits | AES-128 | About $1.1 \times 10^{19}$ years |
| 256 bits | AES-256 | About $3.7 \times 10^{57}$ years |

On average, the key is found **halfway** through the search. Each extra bit **doubles** the time.

Brute force is much more practical against things that **aren't** protected by a hard mathematical problem, like **passwords**:

- **Online guessing**: trying passwords on a **login page**. Slowed down by lockouts and rate limits
- **Offline cracking**: if the attacker has **stolen the password hashes**, they hash guesses on their own computers and compare, with no limits

- **Ways to Speed It Up**:
  - **Dictionary attack**: try common words and leaked passwords first, instead of every combination
  - **Reverse brute force (password spraying)**: try a **few common passwords** against **many accounts**, hoping some match
  - **Credential stuffing**: try usernames and passwords **stolen from another site**, since people reuse them

#### **Birthday Attacks**

Based on the **birthday paradox** from probability theory. Ignoring leap years, there are 365 possible birthdays.

| Question | People Needed |
|----------|---------------|
| Guarantee **some** two people share a birthday | **366** (with 365 days, the 366th person must match someone) |
| 50% chance someone shares **your** birthday (a specific date) | **253** |
| 50% chance **any two** people share a birthday | Only **23** |

**Why only 23?** The comparison is between **pairs** of people, not individuals. With 23 people, there are $\binom{23}{2} = \frac{23 \times 22}{2} = 253$ **pairs**, the same number of chances as the 253 people needed for a specific date.

| People | Chance That Two Share a Birthday |
|--------|----------------------------------|
| 10 | 11.7% |
| 23 | 50.7% |
| 30 | 70.6% |
| 50 | 97.0% |
| 70 | 99.9% |

#### **Birthday Attacks on Hash Functions**

The same idea applies to **hash collisions**. For an $n$-bit hash:

- Finding a message that matches a **specific** hash (**preimage**) takes about $2^n$ tries, like matching a specific birthday
- Finding **any two** messages with the same hash (**collision**) takes only about $2^{n/2}$ tries, like finding any shared birthday

| Hash Size | Collision Effort $2^{n/2}$ | Status |
|-----------|----------------------------|--------|
| 128 bits (MD5) | $2^{64}$ | Too weak |
| 160 bits (SHA-1) | $2^{80}$ | Too weak |
| 256 bits (SHA-256) | $2^{128}$ | Secure |

This is why hash outputs must be **twice as long** as the security level wanted.

#### **Side-Channel Attacks**

**Side-channel attacks** are **non-cryptographic**. They target the **physical implementation** of a cryptosystem, not its maths. A device performing cryptography leaks information through **time, power, radiation and sound**. Many need the attacker to be **near** the device, though some (like timing attacks) can work **over a network**.

| Side Channel | What Leaks | Example |
|--------------|------------|---------|
| **Electromagnetic (TEMPEST)** | Radio emissions from screens, cables and chips | Reconstructing what's on a screen from its emissions, called **Van Eck phreaking** |
| **Power consumption** | How much power the device draws over time | In RSA, **squaring** and **multiplying** draw power differently, so the power trace reveals the **private key's bits** |
| **Timing** | How long operations take | Operations take slightly different times depending on the **private key's bits**, so careful timing reveals them |
| **Acoustic** | Sounds from keyboards or electronics | Telling keys apart by sound to recover a typed **password**, often using **machine learning** |

- **TEMPEST**:
  - A codename for protecting against, and exploiting, **electromagnetic emanations**
  - All electronic devices give off emissions, and specialised equipment can pick them up **from a distance**

- **Power Analysis**:
  - RSA uses **square-and-multiply** (lesson 2.4): each key bit means "square" or "square then multiply"
  - The power trace shows which happened, revealing the key **bit by bit**

- **Timing Attacks**:
  - Work for the same reason as power analysis: different key bits take different amounts of time
  - **Defence**: write **constant-time** code, which takes the same time regardless of the key

- **Countermeasures**:
  - Constant-time code, **random masking** of values during computation, and **shielding** devices
  - Governments and intelligence agencies require their equipment to resist these attacks

#### **Differential Cryptanalysis**

**Differential cryptanalysis**, published by **Biham and Shamir** in 1990, was the first known attack on **DES** better than brute force in theory. It's a **chosen-plaintext** attack.

The idea is to look at **pairs** of plaintexts with a known **difference** (XOR), and track how that difference **changes** from round to round.

Recall a DES (Feistel) round, where $m_0, m_1$ are the two halves of the input:

$$m_{i+1} = m_{i-1} \oplus f(m_i, K_i), \qquad i = 1, 2, \ldots, 16$$

Take two messages $m$ and $m'$ with XOR difference:

$$\Delta m = m \oplus m'$$

For each half:

$$\Delta m_i = m_i \oplus m'_i$$

Then for the next round:

$$
\begin{aligned}
\Delta m_{i+1} &= m_{i+1} \oplus m'_{i+1} \\
&= [m_{i-1} \oplus f(m_i, K_i)] \oplus [m'_{i-1} \oplus f(m'_i, K_i)] \\
&= \Delta m_{i-1} \oplus [f(m_i, K_i) \oplus f(m'_i, K_i)]
\end{aligned}
$$

- **Why It Works**:
  - Inside $f$, the input is XORed with the round key. For both messages, the **key cancels out** in the difference: $(m_i \oplus K_i) \oplus (m'_i \oplus K_i) = m_i \oplus m'_i$
  - So the **input difference** to the S-boxes doesn't depend on the key
  - Certain input differences lead to certain output differences with **high probability**
  - Knowing $\Delta m_{i-1}$ and $\Delta m_i$, the attacker can **predict** $\Delta m_{i+1}$ with high probability, and use the matches to recover key bits

- **Against DES**:
  - Needs about $2^{47}$ **chosen plaintexts** for full 16-round DES
  - Faster than brute force in theory, but **impractical**: collecting that many chosen plaintexts isn't realistic
  - DES's designers already **knew** about this attack, and designed its S-boxes to resist it

#### **Linear Cryptanalysis**

**Linear cryptanalysis**, published by **Matsui** in 1993, is a **known-plaintext** attack, which is easier to arrange than chosen plaintext. It needs fewer texts than differential cryptanalysis: about $2^{43}$ **known plaintexts** for full DES. Matsui **demonstrated** it in practice in 1994, recovering a DES key using 12 computers over about 50 days.

The aim is to find a **linear equation** connecting some plaintext bits, ciphertext bits and key bits:

$$P[\alpha_1, \alpha_2, \ldots, \alpha_a] \oplus C[\beta_1, \beta_2, \ldots, \beta_b] = K[\gamma_1, \gamma_2, \ldots, \gamma_c]$$

- **Notation**:
  - $P[\alpha_1, \ldots, \alpha_a]$: the XOR of the plaintext bits at positions $\alpha_1, \ldots, \alpha_a$
  - $C[\beta_1, \ldots]$ and $K[\gamma_1, \ldots]$: the same for ciphertext and key bits
  - $1 \le a, b \le n$ (the block size) and $1 \le c \le m$ (the key size)
  - The equation holds with probability $p \neq \frac{1}{2}$. The **further $p$ is from $\frac{1}{2}$**, the more useful the equation

- **How It Works**:
  - Compute the left side for many known plaintext-ciphertext pairs
  - If it's **0 more than half the time**, guess that the XOR of those key bits is **0**. If it's **1 more than half the time**, guess **1**
  - Each equation reveals **one bit** of information about the key, and many equations together narrow it down

| Feature | Differential | Linear |
|---------|--------------|--------|
| Published | Biham and Shamir, 1990 | Matsui, 1993 |
| Attacker needs | **Chosen** plaintexts | **Known** plaintexts |
| Texts needed for DES | About $2^{47}$ | About $2^{43}$ |
| Tracks | How **differences** spread through rounds | **Linear relationships** between bits |
| Demonstrated on DES | No (impractical) | Yes (1994) |

AES was designed specifically to **resist both** of these attacks.

<br>

***